In [ ]:

import experiment_helper
import igl
from periodic_simulation_setup import *

h = 2
w = 0.5
avg_len = 0.15
shift = [0, 0]

allowBending = False

y_shift = np.linspace(0, 1, 30)[4]

shift = np.array([5/6, y_shift])
ipu, points, segment_edges, m, markers= periodic_unit_helper.get_shifted_dashline(h, w, avg_len, shift, angle = 0, two_dash = True)

finalMarkers = np.where(np.array(markers) == 1)[0]
m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers)
m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, axis = 1)

fusedVtx = get_fusedVtx_using_markers(len(m.vertices()), finalMarkers)
ipu = inflation.InflatablePeriodicUnit(m, fusedVtx = fusedVtx, epsilon = 1e-9)

viewer = TriMeshViewer(ipu, width=768, height=640)
viewer.showWireframe(True)

# Choose strategy for constraining rigid motion
fixedVars, hessianShift = periodic_unit_helper.get_center_fixedVars(ipu), 0
if not allowBending:
    fixedVars, hessianShift = [ipu.numVars() - 2], 1e-6
else:
    fixedVars, hessianShift = [], 1e-6

ipu.sheet.setUseTensionFieldEnergy(True)
ipu.sheet.setUseHessianProjectedEnergy(False)
ipu.sheet.disableFusedRegionTensionFieldTheory(False)

ipu.sheet.pressure = 2


opts.niter = 500
cr = inflation.inflation_newton(ipu, fixedVars, opts, callback=None, hessianShift = hessianShift)
viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])

az_ipu = get_az_ipu_from_ipu(ipu, m, fusedVtx)
if not allowBending:
    fixedVars, hessianShift = [az_ipu.numVars() - 2], 1e-6
else:
    fixedVars, hessianShift = [], 1e-6
opts.niter = 1000
az_optimizer = inflation.get_inflation_optimizer(az_ipu, fixedVars, opts, callback=None, hessianShift = hessianShift)
cr = az_optimizer.optimize()


In [ ]:
viewer = TriMeshViewer(az_ipu, width=768, height=640)
viewer.showWireframe(True)
viewer.show()


In [ ]:
az_ipu.ipu.get_kappa()

In [ ]:
vars = az_ipu.getVars()
vars[-2] = -0.0001

In [ ]:
az_ipu.setVars(vars)

In [ ]:
az_ipu.gradient()

In [ ]:
max(az_ipu.gradient())

In [ ]:
viewer.update()

In [ ]:
az_ipu.ipu.get_kappa()

In [ ]:
az_ipu.ipu.get_alpha() / np.pi * 180

In [ ]:
fixedVars, hessianShift = [az_ipu.numVars() - 2], 1e-6

fixedVars, hessianShift = [], 1e-6



In [ ]:
framerate = 1 # Update every 5 iterations
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(az_ipu.ipu.sheet)[:, 0])

In [ ]:
opts.niter = 1

In [ ]:
az_optimizer = inflation.get_inflation_optimizer(az_ipu, fixedVars, opts, callback=cb, hessianShift = hessianShift)
cr = az_optimizer.optimize()

In [ ]:
import compute_vibrational_modes

In [ ]:
lambdas, modes = compute_vibrational_modes.compute_vibrational_modes(az_ipu, mtype=compute_vibrational_modes.MassMatrixType.FULL, n=16, sigma=-1e-10, fixedVars = [])


In [ ]:

import mode_viewer, importlib
importlib.reload(mode_viewer);
mview = mode_viewer.ModeViewer(az_ipu, modes, lambdas, amplitude=1000)
# mview.showScalarField(rod_colors)
mview.show()